<a href="https://colab.research.google.com/github/M-Hagemann87/aviation-human-factors/blob/main/aviation_human_factor_preventive_measures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Aviation Human Factor Incidents and Preventive Measures

## Table of Contents:


1. Business Problem
2. Data Audit
3. Feature Engineering
4. Modeling
5. Evaluation
6. Preventive Mapping Layer
7. Deployment
8. Final Conclusions
---

---
# 1. Business Problem


**Dataset:**
NASA Aviation Safety Reporting System (ASRS), ~111,000 incident reports (2005–2025),
filtered to ~40,000 human-factor incidents. The dataset was collected, cleaned, and
preprocessed by the author. Available on Hugging Face and Kaggle.

**Scope:**
This system is designed for decision support, not causal inference. Predictions indicate
which safety interventions are most relevant given an incident narrative — they do not
establish causality.

**Objective:**
Human factors are a primary cause of aviation incidents (e.g., communication failure,
fatigue, distraction, loss of situational awareness). This project aims to:
- Detect human factors from free-text incident narratives
- Structure them as multi-label classification outputs
- Map predictions to actionable preventive measures

**Target Variables (multi-label):**
| Target | Description | Labels |
|---|---|---|
| `human_factors` | Primary human factor type per report | 12 classes |
| `contributing_factors` | Situational and systemic contributors | 17 classes |
| `event_anomaly` | Type of operational anomaly | 29 classes |
| `event_detector` | Who or what first detected the event | 11 classes |

**Input Features:**
- Text: `narratives` — merged from `report_summary` + `report1_narrative`
- Structured: `aircraft1_far_part`

**Modeling Approach:**
- MiniLM (`all-MiniLM-L6-v2`) sentence embeddings (384-dim text representation)
- XGBoost Binary Relevance (one classifier per label, 69 total)
- Class-imbalance handling via `scale_pos_weight`

**Evaluation Metrics:**
- Primary: Macro F1-score (equal weight per label; penalizes rare-class failures)
- Secondary: Precision, Recall per label
- Success threshold: Macro F1 ≥ 0.60 per target is considered operationally viable. Note: not all targets meet this threshold — results discussed in Chapter. 5.

**Validation Strategy:**
- 70/15/15 stratified split (train/val/test)
- Stratification proxy: most frequent `human_factors` label per row
- Validation set used for early stopping; test set held out for final evaluation only

**Preventive Mapping Layer:**
- Rule-based lookup: predicted factors → recommended safety interventions
- Covers labels with frequency >5% in training data
- Low-frequency labels excluded for reliability
- Additive: multiple predicted labels produce a combined recommendation set

**Deployment:**
Gradio application on Hugging Face Spaces for use by aviation safety analysts.

---
# **2. Data Audit**

\- This section performs an initial audit on the dataset, including schema inspection, missing-value analysis, and identification of candidate features for modeling.


### → Action:

- Import core libraries for data analysis and visualization.

In [ ]:
## Core libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### → Action:
- Download the dataset and convert it to a pandas DataFrame.
- A copy of the DataFrame `df_original` was created to preserve the raw data. All subsequent preprocessing and analysis are performed on the working DataFrame `df`.

In [ ]:
import os
import pandas as pd

# Detect Kaggle properly
is_kaggle = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if is_kaggle:
    path = '/kaggle/input/aviation-incident-report-asrs-111k-entries/ASRS-clean-dataset-aviation-safety.parquet'

else:
    !pip install -q kaggle

    os.environ['KAGGLE_CONFIG_DIR'] = "/content/"

    !kaggle datasets download -d mnhagemann21/aviation-incident-report-asrs-111k-entries
    !unzip -o aviation-incident-report-asrs-111k-entries.zip

    # Safe cleanup
    for f in [
        "ASRS-clean-dataset-aviation-safety.csv",
        "ASRS-raw-dataset-aviation-safety.csv",
        "ASRS-raw-dataset-aviation-safety.parquet"
    ]:
        if os.path.exists(f):
            os.remove(f)

    path = "ASRS-clean-dataset-aviation-safety.parquet"

# Parquet path to pandas dataframe
df_original = pd.read_parquet(path)

In [ ]:
# Create a working copy while preserving the raw dataset
df = df_original.copy()

### → Action:

- Initial exploration on the dataframe.

In [ ]:
print(df.shape)
df.info()

### → Action:
- Inspect the rows and columns from the dataframe to overview the data.
- Check for the time coverage.

In [ ]:
df.head(3)


In [ ]:
df["event_datetime"].min(), df["event_datetime"].max()

\- Below; missing value analysis.

In [ ]:
# Generating missing proportion number for each column
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.tail(5)

In [ ]:
missing_pct.head(5)

### ✔ Key Observations
- Dataframe contains 111,492 incident reports with 63 features, from 01/2005 to 12/2025.
- The dataset is already partially cleaned with consistent data types.
- Several columns contain high missing rates (>50%).
- The primary text features are nearly complete:
  - `report1_narrative`
  - `report_summary`

---
## 2.1	Features Selection

\- In this chapter, we evaluate candidate features and filter the dataset to include only human factor reports, aligning the data with the project objective.


#### → Action:
- Verify the class distribution of `primary_problem` to assess imbalance.

In [ ]:
df["primary_problem"].value_counts(normalize=True).head(10)

### ✔ Observations:

- As expected, "human factors" is the dominant class, consistent with the aviation safety literature — human error is the leading cause of incidents.

---
\- Below; we filter the `primary_problem` to retain only human factor incidents, which are the focus of this project.

In [ ]:
df = df[df['primary_problem'] == 'human factors'].copy()

In [ ]:
print("New shape", df.shape)
df["primary_problem"].value_counts()

### ✔ Observations:

- After filtering, the dataframe is smaller remaining approximately 36% of the original size, retaining 40,171 reports.
- All subsequent steps are performed on filtered dataset.

---
### → Action:

  - Inspected individually each column for missing values and content examples, to support informed feature selection decisions.

In [ ]:
# Column Inspection (Missing values + examples)

def inspect_dataframe(df):
    print("DATASET COLUMN INSPECTION\n")

    for col in df.columns:
        missing_count = df[col].isna().sum()
        total = len(df)
        missing_pct = (missing_count / total) * 100

        examples = df[col].dropna().unique()[:5]

        print(f"Column: {col}")
        print(f"Missing values: {missing_count} ({missing_pct:.2f}%)")
        print(f"Examples: {examples}")
        print("-" * 50)

# Run inspection
inspect_dataframe(df)

\- Below; evaluate paired columns (e.g.: person1/2_*, aircraft1/2_*) to determine whether merging is appropriate. The key metric is the complementarity rate: when column 1 is missing, how often does column 2 fill in, and compare with the previous section to check whether the content is viable of merging.

In [ ]:
# Narrative Column Comparison Explorer
### Goal: Decide whether paired columns are worth merging.

from IPython.display import display, HTML
import ipywidgets as widgets

# ── Column pairs to compare ───────────────────────────────────────────────────
COLUMN_PAIRS = [
    ("report1_narrative",    "report2_narrative"),
    ("person1_human_factors","person2_human_factors"),
    ("person1_role",         "person2_role"),
    ("person1_qualification","person2_qualification"),
    ("person1_experience",   "person2_experience"),
    ("person1_location",     "person2_location"),
    ("person1_organization", "person2_organization"),
    ("aircraft1_crew_size" , "aircraft2_crew_size"),
    ("aircraft1_far_part" , "aircraft2_far_part"),
    ("aircraft1_flight_phase" , "aircraft2_flight_phase"),
]

# ── Helper functions ──────────────────────────────────────────────────────────

def pair_stats(df, col1, col2):
    """Return a summary dict for a column pair."""
    has1 = df[col1].notna()
    has2 = df[col2].notna()

    both        = (has1 & has2).sum()
    only1       = (has1 & ~has2).sum()
    only2       = (~has1 & has2).sum()
    neither     = (~has1 & ~has2).sum()

    # When col1 is empty, does col2 fill in?
    complement  = only2                          # rows where col2 saves the day
    total_miss1 = (~has1).sum()
    complement_rate = complement / total_miss1 if total_miss1 > 0 else 0

    return {
        "both"            : both,
        "only_col1"       : only1,
        "only_col2"       : only2,
        "neither"         : neither,
        "total"           : len(df),
        "complement_rows" : complement,
        "complement_rate" : complement_rate,
        "missing_col1"    : total_miss1,
    }


def render_stats(stats, col1, col2):
    """Print a clean summary table."""
    total = stats["total"]
    pct   = lambda n: f"{n:>7,}  ({n/total*100:5.1f}%)"

    print("=" * 60)
    print(f"  PAIR: {col1}  ←→  {col2}")
    print("=" * 60)
    print(f"  Total rows          : {total:,}")
    print(f"  Both filled         : {pct(stats['both'])}")
    print(f"  Only col1 filled    : {pct(stats['only_col1'])}")
    print(f"  Only col2 filled    : {pct(stats['only_col2'])}")
    print(f"  Neither filled      : {pct(stats['neither'])}")
    print()
    print(f"  ── Complementarity ──────────────────────────")
    print(f"  Rows where col1 is missing       : {stats['missing_col1']:,}")
    print(f"  Of those, col2 fills in          : {stats['complement_rows']:,}  "
          f"({stats['complement_rate']*100:.1f}%)")

    verdict = ""
    if stats["complement_rate"] > 0.5:
        verdict = "✅ col2 frequently fills gaps → WORTH MERGING"
    elif stats["complement_rate"] > 0.2:
        verdict = "⚠️  col2 partially fills gaps → CONSIDER merging"
    else:
        verdict = "❌ col2 rarely adds info when col1 is missing → LOW complementarity"
    print(f"\n  Verdict: {verdict}")
    print("=" * 60)


def show_sample_rows(df, col1, col2, case="both", n=3):
    """
    Display n sample rows for a given case:
        'both'    → both columns filled
        'only1'   → only col1 filled
        'only2'   → only col2 filled (complement case)
        'neither' → both empty
    """
    has1 = df[col1].notna()
    has2 = df[col2].notna()

    masks = {
        "both"   : has1 & has2,
        "only1"  : has1 & ~has2,
        "only2"  : ~has1 & has2,
        "neither": ~has1 & ~has2,
    }

    sample = df[masks[case]][[col1, col2]].dropna(how="all").head(n)

    print(f"\n── Sample rows ({case.upper().replace('_',' ')}) ──────────────────")
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\n  [Row {i}]")
        v1 = str(row[col1])[:400] if pd.notna(row[col1]) else "<empty>"
        v2 = str(row[col2])[:400] if pd.notna(row[col2]) else "<empty>"
        print(f"  {col1}:\n    {v1}")
        print(f"  {col2}:\n    {v2}")
    print()


# ── Main runner ───────────────────────────────────────────────────────────────

def run_comparison(df, show_samples=True, sample_n=2):
    """
    Run the full comparison for all column pairs defined in COLUMN_PAIRS.

    Parameters
    ----------
    df          : your working DataFrame (after human-factors filter)
    show_samples: whether to print sample rows
    sample_n    : number of sample rows per case
    """

    # Filter to only pairs where both columns exist in df
    valid_pairs = [(c1, c2) for c1, c2 in COLUMN_PAIRS
                   if c1 in df.columns and c2 in df.columns]

    summary_rows = []

    for col1, col2 in valid_pairs:
        stats = pair_stats(df, col1, col2)
        render_stats(stats, col1, col2)

        if show_samples:
            # Show the most informative case first: complement (only col2)
            for case in ["both", "only2"]:
                show_sample_rows(df, col1, col2, case=case, n=sample_n)

        summary_rows.append({
            "col1"              : col1,
            "col2"              : col2,
            "both_filled_%"     : round(stats["both"]    / stats["total"] * 100, 1),
            "only_col2_%"       : round(stats["only_col2"]/ stats["total"] * 100, 1),
            "complement_rate_%" : round(stats["complement_rate"] * 100, 1),
        })

        print()

    # ── Summary table ─────────────────────────────────────────────────────────
    summary_df = pd.DataFrame(summary_rows)
    print("\n" + "=" * 70)
    print("  MERGE DECISION SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print()
    print("  complement_rate_% = when col1 is missing, how often col2 fills in")
    print("  High complement_rate → strong case for merging with fillna() or concat")

    return summary_df

In [ ]:
summary = run_comparison(df)

\- Based on the pair comparison analysis, the following merges are applied:
  - `report_summary` + `report1_narrative` to; `narrative`
  - `person1_human_factors` + `person2_human_factors` to; `human_factors`

In [ ]:
# Narratives — concatenate (richer NLP input)
df["narratives"] = (
    df["report_summary"].fillna("") + " " +
    df["report1_narrative"].fillna("")
).str.strip()

# Human factors target — union of labels
df["human_factors"] = df.apply(
    lambda r: "; ".join(filter(None, [
        str(r["person1_human_factors"]) if pd.notna(r["person1_human_factors"]) else "",
        str(r["person2_human_factors"]) if pd.notna(r["person2_human_factors"]) else ""
    ])), axis=1
).str.strip("; ")

- Text features were consolidated to maximize information density. The final text feature combines report_summary and report1_narrative to create a richer NLP input.
- Despite low complementarity (~2%), person2_human_factors was merged with person1_human_factors to preserve rare but potentially informative labels in the multi-label target.

### Follow the Feature Selection Decision Summary:
\- All variables were carefully analyzed individually. Merging options were evaluated based on complementarity rates, and highly sparse columns were assessed according to their informative value, missing rate, and risk of introducing misleading content.
- Features were selected based on:
  - Missing rate (<70%)
  - Relevance to human factors
  - Non-redundancy
  - Compatibility with NLP + tabular modeling

- Drop variables:

| Category | Feature | Role | Notes |
|---|---|---|---|
| Dropped | `event_id` | — | Identifier, no predictive value |
| Dropped | `primary_problem` | — | Constant after filtering ("Human Factors") |
| Dropped | `location_*` | — | Not relevant to human factor detection |
| Dropped | High-missing features | — | Removed to reduce noise |
| Dropped | `person2_*` | — | Different person from reporter; low complementarity |
| Dropped | `aircraft2_*` | — | Other aircraft in incident; low signal |
| Dropped | `report1_narrative`, `report_summary` | — | Replaced by merged `narratives` column |
| Dropped | `person1_human_factors`, `person2_human_factors` | — | Replaced by merged `human_factors` target |
| Dropped | `person1_experience` | — | Requires dedicated parsing; deferred |
| Dropped | `aircraft1_flight_phase` and `aircraft2_flight_phase` | — | Not relevant for the project |

- Backed-Up Variables (Kept Outside the Main Dataframe):
  - _Obs.: Back-up variables are stored outside the main dataframe. They are candidates for the Preventive Mapping Layer in a later stage, but excluded from the core modeling pipeline to keep the feature space clean._

| Category | Feature | Role | Notes |
|---|---|---|---|
| Backup | `aircraft1_crew_size` | Context | Crew dynamic context |
| Backup | `person1_communication_breakdown` | Context | 69% missing; strong signal when present but not used as predictor |
| Backup | `event_detection_phase` | Context | Useful for preventive mapping layer |
| Backup | `event_outcome` | Context | Useful for preventive mapping layer |



- Auxiliary Variables (Kept in the Dataframe):

| Category | Feature | Role | Notes |
|---|---|---|---|
| Auxiliary | `event_datetime` | Context | Temporal analysis / optional features |
| Auxiliary | `light_condition` | Context | Environmental context |

- Structured Predictors:

| Category | Feature | Role | Notes |
|---|---|---|---|
| Predictor | `aircraft1_far_part` | Input | Type of operation (commercial, private, etc.) |

- Primary Text Predictor:

| Category | Feature | Role | Notes |
|---|---|---|---|
| Text (Primary) | `narratives` | Input | Merged from `report_summary` + `report1_narrative` |

- Target variables:

| Category | Feature | Role | Notes |
|---|---|---|---|
| Target | `human_factors` | Output | Multi-label — union of `person1` + `person2` human factors |
| Target | `contributing_factors` | Output | Multi-label target |
| Target | `event_anomaly` | Output | Multi-label target |
| Target | `event_detector` | Output | Multi-label target |

In [ ]:
### Variables to keep

pred_text = ["narratives"]

pred_var = ["aircraft1_far_part"]

target_var = ["human_factors",
    "contributing_factors",
    "event_anomaly",
    "event_detector"
    ]

aux_var = ["event_datetime",
    "light_condition",

    ]

### Out of the dataframe

bkp_var = ["aircraft1_crew_size",
    "person1_communication_breakdown",
    "event_detection_phase",
    "event_outcome",
    ]

# New dataframe
df = df[pred_text + pred_var + target_var + aux_var].copy()
print(f"Final shape: {df.shape}")

### ✔ Chapter Conclusions:

- All variables were carefully analyzed individually. Merging options were evaluated based on complementarity rates, and highly sparse columns were assessed according to their informative value, missing rate, and risk of introducing misleading content into the model.
- The final workable dataframe df is now with 8 columns and ~40K reports.
- Primary signals (e.g., person1_*, aircraft1_*) consistently demonstrated strong predictive value. In contrast, secondary entities (person2_*, aircraft2_*) showed low complementarity and minimal incremental signal.

---
# **3.	Feature Engineering**

\- This chapter prepares all features for modeling — targets are normalized and filtered, and predictors are cleaned and encoded.

## 3.1	Target Engineering

\- In this section, composite target labels are decomposed into atomic classes. Frequency distributions are analyzed to assess learnability. Rare categories are grouped into an "Other" class to reduce noise, improve generalization, and stabilize training metrics.

### → Action:

-  We start with multi-label decomposition on all four target columns to discover the real class vocabulary of each.




\- Below; Analysing labels function.

In [ ]:
def analyze_labels(df, col, threshold=0.05):

    counts = (
        df[col]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .loc[lambda s: s != ""]
        .value_counts()
    )

    n_rows        = df[col].notna().sum()
    abs_threshold = threshold * n_rows

    # Stats
    missing = (df[col].isna() | (df[col].str.strip() == "")).sum()

    print(f"  Missing values       : {missing:,}  ({missing/len(df)*100:.1f}%)")
    print(f"  Rows with value      : {n_rows:,}")
    print(f"  Unique labels        : {len(counts)}")
    print(f"  Avg labels / row     : {counts.sum() / n_rows:.2f}")
    print(f"  Threshold ({threshold*100:.0f}%)        : {abs_threshold:,.0f} instances")
    print(f"  Labels kept          : {(counts >= abs_threshold).sum()}")
    print(f"  Labels dropped       : {(counts < abs_threshold).sum()}")
    print()

    # Table
    top = counts.reset_index()
    top.columns = ["label", "count"]
    top["pct"]  = (top["count"] / n_rows * 100).round(1)
    top["keep"] = top["count"] >= abs_threshold
    print(top.head(30).to_string(index=False))

    # Plot
    plot_counts = counts.head(30)
    colors = ["#2196F3" if v >= abs_threshold else "#EF5350" for v in plot_counts.values]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(range(len(plot_counts)), plot_counts.values, color=colors)
    ax.axhline(abs_threshold, color="red", linestyle="--", linewidth=1.2,
               label=f"{threshold*100:.0f}% threshold ({abs_threshold:,.0f})")
    ax.set_xticks(range(len(plot_counts)))
    ax.set_xticklabels(plot_counts.index, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{col}  —  top {len(plot_counts)} labels", fontsize=11, fontweight="bold")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    ax.set_xlabel(
        f"Kept: {(counts >= abs_threshold).sum()} labels  |  "
        f"Dropped (below threshold): {(counts < abs_threshold).sum()} labels",
        fontsize=9
    )
    plt.tight_layout()
    plt.show()

    return counts

\- Below; Drop threshold function (also drop nan).

In [ ]:
def drop_label_threshold(df, col, threshold, drop_rows=True):

    n_rows      = df[col].notna().sum()
    counts      = (
        df[col].dropna()
        .str.split(";").explode()
        .str.strip().str.lower()
        .value_counts()
    )
    drop_labels = set(counts[counts < threshold * n_rows].index)

    # Ensure empty strings are NaN
    df[col] = df[col].replace("", np.nan)

    df[col] = df[col].apply(
        lambda val: "; ".join(
            l.strip() for l in str(val).split(";")
            if l.strip() not in drop_labels
        ) if pd.notna(val) else np.nan
    ).replace("", np.nan)

    if drop_rows:
        before = len(df)
        df = df[df[col].notna()].copy()
        print(f"  Labels dropped : {len(drop_labels)}")
        print(f"  Rows dropped   : {before - len(df):,}")
        print(f"  Rows remaining : {len(df):,}")

    return df

\- Below; Analysing "human_factors" feature.

In [ ]:
counts = analyze_labels(df, "human_factors", threshold=0.03)

- Since `human_factors` has only 12 well-defined labels, no threshold filtering will be applied — all labels will be kept.
- This feature has a high missing value rate (~26.5%, ~10,643 rows). Two options were considered: dropping the missing rows, or relabeling them as "other". Both were rejected — dropping rows would unnecessarily reduce the dataset and discard valid information from other features, while relabeling ~10K rows as "other" would introduce semantic noise and mislead the model with an artificially inflated generic class. Missing values will therefore be kept as NaN, which is the standard approach in multi-label classification — unlabeled rows simply do not contribute to this target during training, but remain available for all other targets.
- Text normalization (lowercase, whitespace stripping, empty string to NaN) will be applied after all target features have been inspected.

---
\- - The "human factors" label appears in 99.9% of rows and **will be removed** —  it is circular (the entire project already filters for human-factor incidents), dominates the distribution (~4× the next label), and adds noise rather than signal to the contributing factors classifier. The remaining 16 labels are kept.

In [ ]:
# Drop only the 7 originally empty rows
df = df[df["contributing_factors"].notna()].copy()
print(f"Rows after drop: {len(df):,}")

In [ ]:
# Remove "human factors" from contributing_factors
def strip_label(val, label_to_remove):
    parts = [p.strip() for p in str(val).split(";") if p.strip().lower() != label_to_remove]
    return "; ".join(parts) if parts else np.nan

df["contributing_factors"] = df["contributing_factors"].apply(
    lambda v: strip_label(v, "human factors")
)
print(f"'human factors' removed from contributing_factors.")

---
\- Below; Analysing "contributing_factors" feature.

In [ ]:
counts = analyze_labels(df, "contributing_factors", threshold=0.03)

- The `"human factors"` label appears in 99.9% of rows and **will be removed**.
It is circular — the entire dataset was already filtered to human-factor incidents, so this label carries no discriminative signal. Its near-constant presence (~4× more frequent than the next label) dominates the distribution, inflates class imbalance, and adds noise to the contributing factors classifier rather than meaningful signal. Removing it reduces the label set from 17 to 16 classes.

- The remaining 16 labels are well-defined and structured — there is no long noisy tail. The 3% frequency threshold will therefore **not** be applied, and all 16 labels are kept. Low-frequency labels such as `manuals`, `staffing`, and `mel` are still operationally meaningful contributing factors, and 250+ occurrences provides sufficient support for a multi-label classifier.

- The 7 rows (0.0%) with missing `contributing_factors` values are negligible and will be dropped — they have no meaningful impact on dataset size or distribution.

---
\- Below; Analysing "event_anomaly" feature.

In [ ]:
counts = analyze_labels(df, "event_anomaly", threshold=0.02)

- The `event_anomaly` presents 560 unique labels, the vast majority (531) are free-text entries manually typed by reporters.
- A 2% frequency threshold (802 instances) will be applied, retaining 29 clean, structured ASRS anomaly codes. This is a data quality decision, not a modeling tradeoff — the dropped labels lack sufficient frequency for reliable learning.
- The 51 missing rows (0.1%) will be dropped.

In [ ]:
# Normalize and drop labels below 2% threshold
df = drop_label_threshold(df, "event_anomaly", threshold=0.02)

---
\- Below; Analysing "event_detector" feature.

In [ ]:
counts = analyze_labels(df, "event_detector", threshold=0.01)

- The `event_detector` feature has only 18 unique labels — a clean, well-structured taxonomy with no free-text noise. A 1% threshold (394 instances) will be applied, retaining 11 labels that cover all operationally meaningful detector types across both human and automation categories.
- The 1% threshold variables will be dropped (7 labels), they are either highly specific to UAS operations — which represent a negligible fraction of the dataset — or roles with very limited involvement in human factor incidents.
- The distinction between human detectors (flight crew, air traffic control, maintenance, etc.) and automation detectors (aircraft ra, terrain warning, air traffic control automation) is preserved in the kept labels, which is valuable for the preventive mapping layer — knowing whether a human or a system first detected the incident is directly actionable information.
- The 714 missing rows (1.8%) will be dropped, since it is a critical feature for the preventive mapping layer and should be as complete and precise as possible. The impact on dataset size is minimal.

In [ ]:
df = drop_label_threshold(df, "event_detector", threshold=0.01)

---
### ✔ Conclusions and Comments:

- Thresholds were set per target based on label cardinality and minimum viable support: higher-cardinality targets (event_anomaly, event_detector) use lower thresholds to retain sufficient label diversity.

## 3.2	Predictive Variables

\- The predictor is inspected, normalized, and mapped to clean categories. Multi-value entries are resolved using domain-driven priority logic. Rare categories are grouped into "other" to reduce noise.

In [ ]:
# Predictor summary: missing rate, unique values, top 3 categories
def audit_predictors(df, cols):
    rows = []
    for col in cols:
        n_missing = df[col].isna().sum()
        n_unique  = df[col].nunique()
        top3      = df[col].value_counts().head(3).index.tolist()
        rows.append({
            "feature"    : col,
            "missing_%"  : round(n_missing / len(df) * 100, 1),
            "unique_vals": n_unique,
            "top_3"      : " | ".join(str(x) for x in top3)
        })
    return pd.DataFrame(rows)

audit_predictors(df, pred_var)

\- Below; For the `aircraft1_far_part` feature, inspect FAA part distribution — extract part numbers only, consolidating all string variants (e.g. "part 121", "other 121").

In [ ]:
vc = (
    df["aircraft1_far_part"]
    .str.extract(r"(\d+)")[0]
    .dropna()
    .astype(int)
    .value_counts()
    .rename_axis("faa_part")
)

pct = (vc / len(df) * 100).round(1)
cumulative = pct.cumsum().round(1)

result = pd.DataFrame({
    "count": vc,
    "pct_%": pct,
    "cumulative_%": cumulative
})

result.head(5)

- The top 3 FAA parts cover ~94% of reports. Each maps to a distinct operational context with known human factor profiles — commercial airlines operate under stricter procedures than general aviation, making this a meaningful signal for multi-label classification.
- FAA Part vs. New Label:
  - part 121 : commercial airline
  - part 91  : general aviation
  - part 135 : air taxi / charter


---
\- Below; Map top 3 parts to readable labels — all others set to "other".

In [ ]:
def map_far_part(val):
    if pd.isna(val):
        return "other"

    # Take first value if multi-entry
    val = str(val).split(";")[0].strip().lower()

    mapping = {
        "part 121": "commercial airline",
        "part 91" : "general aviation",
        "part 135": "air taxi / charter",
    }
    return mapping.get(val, "other")

df["aircraft1_far_part"] = df["aircraft1_far_part"].apply(map_far_part)
df["aircraft1_far_part"].value_counts()

## 3.3	Text Predictor Preprocessing

\- This section prepares the "narratives" column for embedding generation. The raw text is a concatenation of  "report_summary" and "report1_narrative", and contains aviation-specific abbreviations, inconsistent formatting, and noise that must be cleaned before passing to the MiniLM encoder.

In [ ]:
# Sanity check on narratives column before preprocessing
missing = (df["narratives"].isna() | (df["narratives"].str.strip() == "")).sum()
print(f"Missing / empty : {missing:,}")
print(f"Rows with value : {df['narratives'].notna().sum():,}")
print(f"Avg char length : {df['narratives'].dropna().str.len().mean():,.0f}")
print(f"Max char length : {df['narratives'].dropna().str.len().max():,.0f}")
print(f"Min char length : {df['narratives'].dropna().str.len().min():,.0f}")

\- Below; Cleaning the texts.

In [ ]:
import re

def clean_narrative(text):
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove special characters — keep letters, numbers, spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["narratives"] = df["narratives"].apply(clean_narrative)

# Sanity check after cleaning
print(f"Missing / empty : {(df['narratives'] == '').sum():,}")
print(f"Avg char length : {df['narratives'].str.len().mean():,.0f}")
print(f"Sample:")
print(df["narratives"].iloc[0][:300])

- The "narratives" column is now ready for embedding generation. All 37,626 rows have valid content with zero missing values. The average text length decreased slightly from 1,650 to 1,618 characters after cleaning — confirming minimal content loss during normalization.

## 3.4	Target Normalization

\- This section applies final normalization to all target columns — standardizing casing, whitespace, and empty values — then converts semicolon-delimited strings into binary label matrices using MultiLabelBinarizer. This ensures consistent input format for multi-label XGBoost training.

In [ ]:
# Target Normalization + MultiLabelBinarizer

from sklearn.preprocessing import MultiLabelBinarizer

# Step 1: Text normalization on all targets
for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    df[col] = (
        df[col]
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", np.nan)
    )

- Text normalization applied: lowercase, whitespace collapse, and empty-string-to-NaN conversion across all four target columns.

In [ ]:
# Convert semicolon strings to lists
def to_label_list(val):
    if pd.isna(val):
        return []
    return [l.strip() for l in str(val).split(";") if l.strip()]

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    df[col + "_list"] = df[col].apply(to_label_list)

- Semicolon-delimited strings converted to Python lists — required input format for MultiLabelBinarizer.

In [ ]:
# Fit and apply MultiLabelBinarizer per target
mlb = {}

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    mlb[col] = MultiLabelBinarizer()
    matrix = mlb[col].fit_transform(df[col + "_list"])

    print(f"\n  {col}")
    print(f"  Classes : {list(mlb[col].classes_)}")
    print(f"  Shape   : {matrix.shape}")

### ✔ Conclusions:
- All four target columns have been normalized and binarized into binary matrices — human_factors (12 classes), contributing_factors (17 classes), event_anomaly (29 classes), and event_detector (11 classes). Combined, the model will predict across 69 distinct labels. Each row now maps to a consistent binary vector per target, ready for multi-label classification.
  - Note: analyze_labels reports labels above the frequency threshold used for row filtering. The MLB (MultiLabelBinarizer) fits on all labels present in the surviving rows, which may include additional low-frequency labels not dropped at the row level.

## ✔ Chapter Conclusion:
This chapter transformed the raw filtered dataset (40,171 rows, 8 columns) into a modeling-ready structure of 37,626 rows. The reduction of ~6.3% resulted from dropping rows with missing or below-threshold labels — a deliberate quality decision, not information loss. All targets were decomposed, filtered, and binarized into 69 distinct binary labels. All structured predictors were normalized into compact categorical groups using domain-driven priority logic. The primary text predictor was cleaned with minimal content loss. The dataset is now ready for the next Chapter: train/test split, embedding generation, and multi-label XGBoost training.

---
# **4.	Modeling**

\- This chapter covers the training pipeline: train/test split, MiniLM embedding generation, structured feature encoding, and XGBoost multi-label classification. Independent XGBoost classifiers are trained per target label — all sharing the same input feature space while preserving task-specific specialization.

---
### 4.1 Train/Test Split
- The split preserves class proportions across all three partitions — train (70%), validation (15%), and test (15%) distributions are closely aligned for human_factors, with no label deviating more than ~0.6pp between splits. This confirms the stratification proxy is effective. The validation set will be used for hyperparameter tuning; the test set is held out strictly for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Stratification key for multi-label target ─────────────────────────────────
# Use the most frequent human_factors label per row as proxy for stratification.
# Rows with no human_factors labels use "none" as key.

hf_cols = mlb["human_factors"].classes_

# Build binary matrix for human_factors
hf_matrix = mlb["human_factors"].transform(df["human_factors_list"])

# Assign stratification key: first active label (ordered by global frequency)
hf_freq_order = (
    pd.Series(hf_matrix.sum(axis=0), index=hf_cols)
    .sort_values(ascending=False)
    .index.tolist()
)

def get_strat_key(row_vector):
    for label in hf_freq_order:
        idx = list(hf_cols).index(label)
        if row_vector[idx] == 1:
            return label
    return "none"

df["_strat_key"] = [get_strat_key(row) for row in hf_matrix]

print("Stratification key distribution:")
print(df["_strat_key"].value_counts())

In [ ]:
# 70 / 15 / 15 split

# First split: 70% train, 30% temp
df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["_strat_key"]
)

# Second split: 50/50 on temp → 15% val, 15% test
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp["_strat_key"]
)

# Drop helper column
for split in [df_train, df_val, df_test]:
    split.drop(columns=["_strat_key"], inplace=True)
df.drop(columns=["_strat_key"], inplace=True)

print(f"  Train : {len(df_train):,}  ({len(df_train)/len(df)*100:.1f}%)")
print(f"  Val   : {len(df_val):,}  ({len(df_val)/len(df)*100:.1f}%)")
print(f"  Test  : {len(df_test):,}  ({len(df_test)/len(df)*100:.1f}%)")

In [ ]:
# ── Sanity check: class distribution across splits ────────────────────────────

def check_label_distribution(df_tr, df_v, df_ts, col="_list_col"):
    """Compare label frequencies across splits for a given target."""
    from collections import Counter

    def label_pcts(series):
        counts = Counter(l for labels in series for l in labels)
        total = sum(counts.values())
        return {k: round(v / total * 100, 1) for k, v in counts.items()}

    tr = label_pcts(df_tr)
    vl = label_pcts(df_v)
    ts = label_pcts(df_ts)

    all_labels = sorted(set(tr) | set(vl) | set(ts))
    rows = []
    for label in all_labels:
        rows.append({
            "label": label,
            "train_%": tr.get(label, 0),
            "val_%": vl.get(label, 0),
            "test_%": ts.get(label, 0),
        })
    return pd.DataFrame(rows)

dist = check_label_distribution(
    df_train["human_factors_list"],
    df_val["human_factors_list"],
    df_test["human_factors_list"]
)
print("human_factors — label distribution across splits:")
print(dist.to_string(index=False))

### Observations:
- The split preserves class proportions across all three partitions — train, validation, and test distributions are closely aligned for human_factors, confirming the stratification proxy is effective. The validation set will be used for hyperparameter tuning, and the test set is held out strictly for final evaluation.

---
### 4.2 Feature Encoding
- This section transforms all predictors into numeric representations suitable for XGBoost. The text predictor (narratives) is encoded into dense vectors using a pretrained MiniLM sentence transformer. The structured predictor (aircraft1_far_part) is one-hot encoded. Both are concatenated into a single feature matrix per split.

---
\- Below;  encode each narrative into a 384-dimensional dense vector using pretrained MiniLM. No fitting on training data is required, so applying it to all splits does not introduce data leakage.

In [ ]:
# MiniLM encodes each narrative into a 384-dim dense vector.
# The model is pretrained — no fitting on training data is required,
# so applying it to all splits does not introduce data leakage.

!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
encoder = SentenceTransformer(model_name)

print(f"Model: {model_name}")
print(f"Embedding dim: {encoder.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings per split
# MiniLM truncates inputs at 256 tokens — longer narratives are clipped automatically.

emb_train = encoder.encode(df_train["narratives"].tolist(), show_progress_bar=True, batch_size=128)
emb_val   = encoder.encode(df_val["narratives"].tolist(),   show_progress_bar=True, batch_size=128)
emb_test  = encoder.encode(df_test["narratives"].tolist(),  show_progress_bar=True, batch_size=128)

print(f"  Train embeddings : {emb_train.shape}")
print(f"  Val embeddings   : {emb_val.shape}")
print(f"  Test embeddings  : {emb_test.shape}")

- All three splits encoded successfully. Batch size 128 balances memory usage and speed on Colab's GPU runtime.
---
\- Below; one-hot encode the 5 structured predictors. Fit on training set only to prevent data leakage.

In [ ]:
# Structured Predictor Encoding
# One-hot encode FAR Part (only structured predictor)
# Fit on train only — val/test use the same encoder to prevent leakage.

from sklearn.preprocessing import OneHotEncoder

struct_cols = ["aircraft1_far_part"]

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Fit ONLY on train
ohe.fit(df_train[struct_cols])

# Transform all splits
struct_train = ohe.transform(df_train[struct_cols])
struct_val   = ohe.transform(df_val[struct_cols])
struct_test  = ohe.transform(df_test[struct_cols])

print(f"Structured features: {struct_train.shape[1]} columns")
print(f"Categories: {ohe.get_feature_names_out().tolist()}")

- The predictor expanded into 4 one-hot columns. The handle_unknown="ignore" parameter ensures unseen categories at inference time produce a zero-vector rather than an error.
---
\- Below; concatenate embeddings and structured features into a single feature matrix per split.

In [ ]:
# Feature Matrix Assembly
# Concatenate MiniLM embeddings (384-dim) + one-hot structured features
# into a single feature matrix per split.

X_train = np.concatenate([emb_train, struct_train], axis=1)
X_val   = np.concatenate([emb_val,   struct_val],   axis=1)
X_test  = np.concatenate([emb_test,  struct_test],  axis=1)

print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  ── Breakdown ──")
print(f"  MiniLM embeddings : {emb_train.shape[1]} dims")
print(f"  Structured OHE    : {struct_train.shape[1]} dims")
print(f"  Total features    : {X_train.shape[1]}")

###✔ Observation:

The final feature matrix has 388 dimensions per row (384 MiniLM + 4 OHE) across all three splits. The encoder objects (encoder, ohe) and the mlb binarizers from Chapter 3 must be preserved for the deployment pipeline.

---
### 4.3 Model Training

This section trains independent XGBoost (Extreme Gradient Boosting) binary
classifiers for each label across all four target variables — a BR (Binary
Relevance) strategy. Each classifier learns to predict a single label (present/absent) from the shared 388-feature input matrix. Early stopping
on the validation set prevents overfitting.

A `RETRAIN` flag controls execution mode:
- `RETRAIN = False` — loads pre-trained models from disk. Use for evaluation,
  feature importance analysis, and deployment work.
- `RETRAIN = True` — trains all 69 classifiers from scratch. Required only
  when the feature space or training data changes.

---
→ Action:
  - Set the `RETRAIN` flag and load pre-trained models from disk if available.

In [ ]:
# Re-fit MLB on train data only to eliminate label leakage from test/val.
# Replaces the preliminary fit done above for stratification.
for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    mlb[col] = MultiLabelBinarizer()
    mlb[col].fit(df_train[col + "_list"])
print("MLB re-fitted on train data only.")

In [ ]:
import pickle
import os
from collections import defaultdict

# Retrain flag
# Set to True to retrain all 69 classifiers from scratch.
# Set to False to load pre-trained models from disk (saves ~15–20 min on T4).

RETRAIN = False

MODELS_PATH = "/content/drive/MyDrive/DS_Projects/Models_Trained/export/xgb_models.pkl"

if not RETRAIN:
    if os.path.exists(MODELS_PATH):
        with open(MODELS_PATH, "rb") as f:
            models = defaultdict(dict, pickle.load(f))
        print(f"✔ Models loaded from: {MODELS_PATH}")
        print(f"  Targets  : {list(models.keys())}")
        print(f"  Classifiers loaded: {sum(len(v) for v in models.values())}")
    else:
        raise FileNotFoundError(
          f"MODELS_PATH not found: {MODELS_PATH}\n"
          "Set RETRAIN = True to train from scratch."
        )

→ Action:
- Build binary target matrices (Y) for train, validation, and test sets using
  the fitted MLB (MultiLabelBinarizer) objects from Chapter 3.
- Always runs regardless of the `RETRAIN` flag — Y matrices are required for
  evaluation in Chapter 5.

In [ ]:
# Build Y matrices per target per split

Y_train, Y_val, Y_test = {}, {}, {}

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    Y_train[col] = mlb[col].transform(df_train[col + "_list"])
    Y_val[col]   = mlb[col].transform(df_val[col + "_list"])
    Y_test[col]  = mlb[col].transform(df_test[col + "_list"])

    print(f"  {col:<25s}  train {Y_train[col].shape}  val {Y_val[col].shape}  test {Y_test[col].shape}")

\- Y matrices confirmed: 69 total binary labels across all four targets,
  consistent with the binarization in Chapter 3.

---

→ Action:
- Train one XGBoost (Extreme Gradient Boosting) classifier per label using
  BR (Binary Relevance). Early stopping monitors log-loss on the validation
  set (patience = 10 rounds). A scale_pos_weight (SPW) is computed per label
  to handle class imbalance — rare labels receive proportionally higher weight
  to improve recall.
- Skipped when `RETRAIN = False`.

In [ ]:
# Skipped when RETRAIN = False — models already loaded above.

if RETRAIN:

    # Train XGBoost (Extreme Gradient Boosting) classifiers — BR (Binary Relevance)

    import xgboost as xgb
    from collections import defaultdict

    models = defaultdict(dict)

    xgb_params = {
        "objective"             : "binary:logistic",
        "eval_metric"           : "logloss",
        "tree_method"           : "hist",
        "learning_rate"         : 0.1,
        "max_depth"             : 6,
        "n_estimators"          : 300,
        "early_stopping_rounds" : 10,
        "verbosity"             : 0,
        "random_state"          : 42,
    }

    total_labels = sum(Y_train[col].shape[1] for col in Y_train)
    trained = 0

    for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
        n_labels    = Y_train[col].shape[1]
        label_names = mlb[col].classes_

        print(f"\n── {col} ({n_labels} labels) ──")

        for i, label in enumerate(label_names):
            y_tr = Y_train[col][:, i]
            y_vl = Y_val[col][:, i]

            n_pos = y_tr.sum()
            n_neg = len(y_tr) - n_pos
            spw   = n_neg / n_pos if n_pos > 0 else 1.0

            clf = xgb.XGBClassifier(**xgb_params, scale_pos_weight=spw)
            clf.fit(X_train, y_tr, eval_set=[(X_val, y_vl)], verbose=False)

            models[col][label] = clf
            trained += 1

            print(f"  [{i+1:>2}/{n_labels}] {label:<45s}  "
                  f"best_iter={clf.best_iteration:<4d}  "
                  f"val_logloss={clf.best_score:.4f}  "
                  f"spw={spw:.1f}")

    print(f"\n✔ Training complete: {trained} / {total_labels} classifiers trained.")

### ✔ Observation:

- **If `RETRAIN = False`:** All 69 pre-trained classifiers loaded successfully
  from disk. Model objects are identical to those produced during the original
  training run — all subsequent evaluation and feature importance steps are
  fully reproducible without retraining.

- **If `RETRAIN = True`:** All 69 binary classifiers trained successfully with
  early stopping. The SPW (scale_pos_weight) parameter compensates for label
  imbalance — labels with fewer positive examples receive proportionally higher
  weight, preventing the model from trivially predicting the majority class.
  The best_iteration per classifier indicates how many boosting rounds were
  used before validation loss plateaued, typically well below the 300-round
  cap, confirming early stopping is effective. All trained models are stored
  in the `models` dictionary, keyed by target name and label.

---
---
### 4.4 Feature Importance

This section quantifies the contribution of each feature group — MiniLM
(Miniature Language Model) embeddings vs. OHE (One-Hot Encoded) structured
predictors — to the trained models. XGBoost (Extreme Gradient Boosting)
computes a feature importance score for every classifier during training.
By aggregating these scores across all 69 classifiers, we obtain a global
picture of what the model relies on.

This analysis directly informs the deployment design decision in Chapter 7:
if structured features contribute negligible signal, exposing them in the
inference UI adds user friction with no meaningful gain.

→ Action:
- Aggregate XGBoost (Extreme Gradient Boosting) feature importances across
  all 69 classifiers.
- Separate total importance into two groups: MiniLM embedding dimensions
  (384 features) vs. OHE structured predictors (4 features).
- Report the percentage split and identify the top contributing structured
  features.

In [ ]:
# ── Feature names ─────────────────────────────────────────────────────────────
embedding_names = [f"emb_{i}" for i in range(384)]
ohe_names       = ohe.get_feature_names_out(struct_cols).tolist()
all_feature_names = embedding_names + ohe_names

# ── Aggregate importance across all 69 classifiers ───────────────────────────
importance_sum = np.zeros(len(all_feature_names))

for col in models:
    for label, clf in models[col].items():
        importance_sum += clf.feature_importances_

importance_df = pd.DataFrame({
    "feature"   : all_feature_names,
    "importance": importance_sum
}).sort_values("importance", ascending=False).reset_index(drop=True)

# ── Embedding vs. structured split ───────────────────────────────────────────
emb_mask     = importance_df["feature"].str.startswith("emb_")
emb_total    = importance_df.loc[emb_mask,  "importance"].sum()
struct_total = importance_df.loc[~emb_mask, "importance"].sum()
total        = emb_total + struct_total

print(f"  MiniLM (Miniature Language Model) embeddings : {emb_total/total*100:.1f}%")
print(f"  OHE (One-Hot Encoded) structured predictors : {struct_total/total*100:.1f}%")
print(f"\n  Top 10 structured features by importance:")
print(
    importance_df[~emb_mask]
    .head(10)
    .assign(pct=lambda d: (d["importance"] / total * 100).round(2))
    .to_string(index=False)
)

→ Action:
- Visualize the embedding vs. structured importance split as a vertical bar chart.
- Plot the top 15 OHE (One-Hot Encoded) structured features as a horizontal bar
  chart, sorted by aggregated importance.

In [ ]:
struct_df = (
    importance_df[~emb_mask]
    .head(15)
    .sort_values("importance", ascending=True)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Embedding vs. structured split ────────────────────────────────────
ax = axes[0]
groups = ["MiniLM\nEmbeddings", "Structured\nPredictors"]
values = [emb_total / total * 100, struct_total / total * 100]
colors = ["#2196F3", "#FFA726"]

bars = ax.bar(groups, values, color=colors, width=0.45, edgecolor="none")

for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_ylim(0, 110)
ax.set_ylabel("% of Total Aggregated Importance")
ax.set_title("Feature Group Contribution\n(aggregated across 69 classifiers)",
             fontsize=11, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

# ── Plot 2: Top structured features ──────────────────────────────────────────
ax = axes[1]
colors_struct = ["#EF5350" if v < struct_df["importance"].max() * 0.3
                 else "#FFA726" if v < struct_df["importance"].max() * 0.6
                 else "#66BB6A"
                 for v in struct_df["importance"]]

ax.barh(struct_df["feature"], struct_df["importance"],
        color=colors_struct, edgecolor="none")
ax.set_xlabel("Aggregated Feature Importance")
ax.set_title("Top 15 OHE (One-Hot Encoded) Structured Features\n"
             "(aggregated across 69 classifiers)",
             fontsize=11, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### ✔ Conclusions:

- The model relies primarily on narrative text (97.8%), confirming that incident descriptions encode the most critical safety signals.
- Among structured features, general aviation and commercial airline FAR Part categories contribute the most, reflecting distinct human factor risk profiles between operational contexts.
- The 2.2% structured contribution is small but meaningful — both top FAR Part categories show positive importance, justifying their inclusion in the pipeline.


---
# **5.	Evaluation**

Evaluation on the held-out test set across all 69 classifiers. Primary metric: **Macro F1** (equal weight per label, strict on rare classes). Secondary: Micro F1 (instance-weighted, reflects practical throughput) and Weighted F1. Reference threshold: **Macro F1 ≥ 0.60**.

### 5.1 Test Set Predictions

\- Below; generate binary predictions and probability scores on the test set for all 69 classifiers. Probabilities are retained for confidence-filtered deployment in Chapter 6.

In [ ]:
# Predictions on test set

Y_pred = {}
Y_prob = {}

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    label_names = mlb[col].classes_
    n_labels = len(label_names)

    pred_matrix = np.zeros((len(X_test), n_labels))
    prob_matrix = np.zeros((len(X_test), n_labels))

    for i, label in enumerate(label_names):
        clf = models[col][label]
        pred_matrix[:, i] = clf.predict(X_test)
        prob_matrix[:, i] = clf.predict_proba(X_test)[:, 1]

    Y_pred[col] = pred_matrix.astype(int)
    Y_prob[col] = prob_matrix

    print(f"  {col:<25s}  predictions {Y_pred[col].shape}  probabilities {Y_prob[col].shape}")

\- All predictions generated. Binary predictions and probability scores stored for each target.

### 5.2 Per-Target Metrics

- Below; compute Macro F1, Micro F1, Weighted F1, Precision, and Recall per target on the test set. Reporting all three F1 variants together exposes the gap between overall instance-level performance (Micro) and rare-label performance (Macro).

In [ ]:
# Per-target evaluation — Macro, Micro, and Weighted F1

from sklearn.metrics import f1_score, precision_score, recall_score

target_results = []

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    y_true = Y_test[col]
    y_pred = Y_pred[col]

    macro_f1    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    micro_f1    = f1_score(y_true, y_pred, average="micro",    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_p     = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_r     = recall_score(y_true, y_pred,    average="macro", zero_division=0)

    target_results.append({
        "target"      : col,
        "n_labels"    : y_true.shape[1],
        "macro_f1"    : round(macro_f1,    4),
        "micro_f1"    : round(micro_f1,    4),
        "weighted_f1" : round(weighted_f1, 4),
        "macro_prec"  : round(macro_p,     4),
        "macro_rec"   : round(macro_r,     4),
    })

    print(f"  {col:<25s}  macro={macro_f1:.4f}  micro={micro_f1:.4f}  weighted={weighted_f1:.4f}  "
          f"P={macro_p:.4f}  R={macro_r:.4f}")

results_df = pd.DataFrame(target_results)
print(f"\n{results_df.to_string(index=False)}")

### ✔ Per-Target Results

| Target | Macro F1 | Micro F1 | Gap |
|---|---|---|---|
| human_factors | 0.363 | 0.520 | +0.157 |
| contributing_factors | 0.326 | 0.464 | +0.138 |
| event_anomaly | 0.580 | 0.666 | +0.086 |
| event_detector | 0.578 | 0.742 | +0.164 |

- The Macro–Micro gap confirms a long-tail problem, not a pipeline failure. `event_anomaly` / `event_detector` approach threshold — their labels are objective and consistently reported. `human_factors` / `contributing_factors` fall short — their labels reflect internal mental states and causal attributions, inherently ambiguous in voluntary safety reports.

### 5.3 Per-Label Breakdown
- Below; compute F1, Precision, Recall, and support for every individual label across all four targets. This identifies specific failure modes and supports the mapping exclusion decisions in Chapter 6.

In [ ]:
# Per-label evaluation

from sklearn.metrics import classification_report

for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
    y_true = Y_test[col]
    y_pred = Y_pred[col]
    label_names = mlb[col].classes_

    print(f"\n{'='*70}")
    print(f"  {col.upper()}  ({len(label_names)} labels)")
    print(f"{'='*70}")
    print(classification_report(
        y_true, y_pred,
        target_names=label_names,
        zero_division=0,
        digits=3
    ))

### ✔ Per-Label Analysis

Three failure modes explain the low-scoring labels:

- **Catch-all labels** (`other / unknown` F1=0.03, `less severe` F1=0.25): residual by definition, near-zero recall is correct. Excluded from the mapping layer.
- **Rare labels** (6 `contributing_factors` labels, support <100, avg F1=0.155): data limitation, not tunable. `fatigue` (P=0.58, R=0.26) additionally reflects known under-reporting in voluntary safety systems.
- **Overlapping labels** (`confusion`, `distraction`, `time pressure`, `workload`, F1 0.24–0.36): co-occur and share vocabulary; partial signal learned but under-prediction is the safer failure direction for a decision-support tool.

Conversely, labels with objective definitions score well: `procedural hazardous material violation` (0.933), `conflict nmac` (0.854), `person flight crew` (0.817).

### 5.4 Performance Visualization
\- Below; three complementary plots:
1. **Per-label F1 bar charts** (one per target) — sorted ascending, colour-coded by threshold.
2. **Macro vs. Micro F1 comparison** — visualises the gap across all four targets.
3. **Support vs. F1 scatter** — tests the relationship between label frequency and model performance.

In [ ]:
# ── 5.4 Performance Visualisation ────────────────────────────────────────────

from sklearn.metrics import f1_score as f1_per_label
import matplotlib.pyplot as plt
import numpy as np

TARGET_COLS = ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]

# ── Pre-compute per-label F1 and support for all targets ──────────────────────
label_f1s    = {}   # {col: array of f1 per label}
label_sup    = {}   # {col: array of support per label}
label_names_ = {}   # {col: list of label names}

for col in TARGET_COLS:
    y_true = Y_test[col]
    y_pred = Y_pred[col]
    names  = mlb[col].classes_

    f1s = np.array([f1_per_label(y_true[:, i], y_pred[:, i], zero_division=0)
                    for i in range(len(names))])
    sup = y_true.sum(axis=0).astype(int)

    label_f1s[col]    = f1s
    label_sup[col]    = sup
    label_names_[col] = names

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — Per-label F1 bar charts (2×2 grid)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, col in enumerate(TARGET_COLS):
    f1s   = label_f1s[col]
    names = label_names_[col]
    sup   = label_sup[col]

    order         = np.argsort(f1s)
    sorted_names  = [names[i]  for i in order]
    sorted_f1s    = [f1s[i]    for i in order]
    sorted_sup    = [sup[i]    for i in order]

    colors = ["#EF5350" if f < 0.4 else "#FFA726" if f < 0.6 else "#66BB6A"
              for f in sorted_f1s]

    ax = axes[idx]
    bars = ax.barh(range(len(sorted_names)), sorted_f1s, color=colors, edgecolor="none")

    # Annotate support count on each bar
    for j, (bar, s) in enumerate(zip(bars, sorted_sup)):
        ax.text(min(bar.get_width() + 0.01, 0.97), bar.get_y() + bar.get_height() / 2,
                f"n={s}", va="center", fontsize=6, color="#555555")

    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names, fontsize=7)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{col}  ({len(names)} labels)", fontsize=11, fontweight="bold")
    ax.axvline(0.6, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="0.60 threshold")
    ax.legend(fontsize=7, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Per-Label F1 Scores — All Targets\n(support count shown on each bar)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — Macro vs. Micro vs. Weighted F1 comparison
# ─────────────────────────────────────────────────────────────────────────────

macro_f1s    = [results_df.loc[results_df["target"] == col, "macro_f1"].values[0]    for col in TARGET_COLS]
micro_f1s    = [results_df.loc[results_df["target"] == col, "micro_f1"].values[0]    for col in TARGET_COLS]
weighted_f1s = [results_df.loc[results_df["target"] == col, "weighted_f1"].values[0] for col in TARGET_COLS]

x      = np.arange(len(TARGET_COLS))
width  = 0.25
labels = [c.replace("_", "\n") for c in TARGET_COLS]

fig, ax = plt.subplots(figsize=(11, 5))

b1 = ax.bar(x - width, macro_f1s,    width, label="Macro F1",    color="#EF5350", edgecolor="none")
b2 = ax.bar(x,         micro_f1s,    width, label="Micro F1",    color="#2196F3", edgecolor="none")
b3 = ax.bar(x + width, weighted_f1s, width, label="Weighted F1", color="#66BB6A", edgecolor="none")

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

ax.axhline(0.60, color="black", linestyle="--", linewidth=1.0, alpha=0.5, label="0.60 reference")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 0.85)
ax.set_ylabel("F1 Score")
ax.set_title("Macro vs. Micro vs. Weighted F1 — Per Target\n"
             "(gap between Macro and Micro/Weighted reveals long-tail label problem)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — Support vs. F1 scatter (label-level, all targets combined)
# ─────────────────────────────────────────────────────────────────────────────
target_colors = {"human_factors": "#EF5350", "contributing_factors": "#FFA726",
                 "event_anomaly": "#2196F3",  "event_detector": "#66BB6A"}

fig, ax = plt.subplots(figsize=(10, 6))

for col in TARGET_COLS:
    f1s  = label_f1s[col]
    sup  = label_sup[col]
    ax.scatter(sup, f1s, color=target_colors[col], alpha=0.75, s=55,
               label=col.replace("_", " "), edgecolors="none")

ax.axhline(0.60, color="gray", linestyle="--", linewidth=0.9, alpha=0.6, label="0.60 threshold")
ax.set_xscale("log")
ax.set_xlabel("Label Support (log scale) — positive instances in test set")
ax.set_ylabel("F1 Score")
ax.set_title("Label Support vs. F1 Score — All 69 Labels\n"
             "(low support is the primary predictor of poor F1)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## ✔ Chapter 5 — Conclusion

`event_anomaly` and `event_detector` approach Macro F1 0.60 (Micro: 0.67 / 0.74). `human_factors` and `contributing_factors` fall short due to catch-all labels, rare-label data scarcity, and semantic overlap — properties of the data, not the pipeline. The Chapter 6 confidence filter (≥0.50) trades recall for precision, the correct direction for a tool where spurious recommendations erode analyst trust faster than missed ones.

---
# **6.	Preventive Mapping Layer**

This chapter builds a rule-based mapping layer that connects model predictions to actionable preventive recommendations. The mapping is not learned — it is a deterministic lookup table designed by domain logic, linking predicted human factors and contributing factors to specific safety interventions. This layer serves as the decision-support output of the system.

### 6.1 Mapping Design
- The mapping follows three design principles:

  - Only labels with frequency >5% in the training set are mapped — rare labels lack sufficient representation for reliable recommendations.
  - Recommendations are drawn from established aviation safety practices (CRM training, fatigue risk management, SOPs, etc.).
  - The layer is additive — multiple predicted labels produce a combined set of recommendations, not a single fixed output.


\- Below; define the preventive measure mappings for human_factors and contributing_factors. Each label maps to a list of recommended interventions.

In [ ]:
# Preventive Measures Mapping

human_factors_mapping = {
    "situational awareness": [
        "Reinforce Crew Resource Management (CRM) training",
        "Implement cross-check and callout protocols",
        "Review cockpit workload distribution procedures",
    ],
    "communication breakdown": [
        "Standardize read-back / hear-back protocols",
        "Reinforce sterile cockpit procedures",
        "Review ATC-crew communication training",
    ],
    "time pressure": [
        "Review turnaround and dispatch scheduling policies",
        "Train crews on decision-making under time constraints",
        "Implement mandatory go/no-go checklists",
    ],
    "confusion": [
        "Improve signage, markings, and chart clarity",
        "Standardize procedure naming conventions",
        "Reinforce briefing and debriefing protocols",
    ],
    "distraction": [
        "Enforce sterile cockpit / sterile maintenance area rules",
        "Review interruption management training",
        "Implement task-resumption checklists",
    ],
    "training / qualification": [
        "Conduct recurrent training gap analysis",
        "Review type-rating and qualification currency requirements",
        "Implement mentorship or supervised practice programs",
    ],
    "workload": [
        "Review staffing levels and task distribution",
        "Implement workload assessment tools in pre-flight briefing",
        "Train crews on workload shedding techniques",
    ],
    "troubleshooting": [
        "Standardize troubleshooting flowcharts and decision trees",
        "Review maintenance manual accessibility and clarity",
        "Reinforce systematic fault-isolation procedures",
    ],
    "human-machine interface": [
        "Review cockpit / maintenance panel ergonomics",
        "Improve automation mode awareness training",
        "Conduct interface usability audits",
    ],
    "fatigue": [
        "Review duty time and rest requirements compliance",
        "Implement Fatigue Risk Management System (FRMS)",
        "Train crews on fatigue self-assessment techniques",
    ],
}

contributing_factors_mapping = {
    "aircraft": [
        "Review aircraft maintenance and inspection cycles",
        "Conduct fleet-wide reliability trend monitoring",
    ],
    "chart or publication": [
        "Audit chart and publication currency and accuracy",
        "Standardize revision distribution and acknowledgment procedures",
    ],
    "company policy": [
        "Review and update SOPs for alignment with operational reality",
        "Conduct safety culture and compliance audits",
    ],
    "weather": [
        "Review weather decision-making training and tools",
        "Improve weather briefing integration into dispatch workflow",
    ],
    "logbook entry": [
        "Standardize logbook documentation procedures",
        "Implement logbook entry verification protocols",
    ],
    "environment - non weather related": [
        "Review airport environment hazard assessments",
        "Improve wildlife, lighting, and terrain awareness programs",
    ],
    "procedure": [
        "Conduct SOP gap analysis and update cycle",
        "Reinforce procedural compliance monitoring",
    ],
}

print(f"  Human factors mapped    : {len(human_factors_mapping)} labels")
print(f"  Contributing factors mapped : {len(contributing_factors_mapping)} labels")
print(f"  Total recommendations  : "
      f"{sum(len(v) for v in human_factors_mapping.values()) + sum(len(v) for v in contributing_factors_mapping.values())}")

- Mappings defined for the most frequent human factors and contributing factors labels. Low-frequency labels (physiological - other, other / unknown, and contributing factors below 5%) are excluded — their predictions lack the consistency needed for reliable recommendations.

### 6.2 Mapping Function

- Below; implement the function that takes model predictions and returns a combined set of preventive recommendations. The function accepts both binary predictions and probability scores — a confidence threshold filters low-confidence predictions before mapping.

In [ ]:
# Recommendation generator

def get_recommendations(predictions, probabilities=None, confidence_threshold=0.5):
    """
    Generate preventive recommendations from model predictions.

    Parameters
    ----------
    predictions : dict
        Binary prediction arrays keyed by target name.
    probabilities : dict, optional
        Probability arrays keyed by target name. If provided,
        only predictions above confidence_threshold are used.
    confidence_threshold : float
        Minimum probability to include a prediction (default 0.5).

    Returns
    -------
    dict with keys:
        - 'detected_factors': list of predicted labels
        - 'recommendations': deduplicated list of preventive measures
        - 'unmapped_factors': predicted labels with no mapping
    """
    detected = []
    recommendations = []
    unmapped = []

    mappings = {
        "human_factors": human_factors_mapping,
        "contributing_factors": contributing_factors_mapping,
    }

    for col in ["human_factors", "contributing_factors"]:
        if col not in predictions:
            continue

        pred = predictions[col]
        prob = probabilities[col] if probabilities and col in probabilities else None
        label_names = mlb[col].classes_

        for i, label in enumerate(label_names):
            if pred[i] == 1:
                # Filter by confidence if probabilities are available
                if prob is not None and prob[i] < confidence_threshold:
                    continue

                detected.append(label)

                if label in mappings[col]:
                    recommendations.extend(mappings[col][label])
                else:
                    unmapped.append(label)

    # Deduplicate recommendations while preserving order
    seen = set()
    unique_recs = []
    for rec in recommendations:
        if rec not in seen:
            seen.add(rec)
            unique_recs.append(rec)

    return {
        "detected_factors": detected,
        "recommendations": unique_recs,
        "unmapped_factors": unmapped,
    }

print("Recommendation function defined.")

- The function is deterministic and stateless — it can be called independently per report at inference time. The confidence_threshold parameter provides a tunable knob for balancing recommendation coverage vs. precision in deployment.

### 6.3 Mapping Validation
- Below; apply the mapping to the test set predictions and summarize coverage — how many test reports receive at least one recommendation, and how frequently each recommendation appears.

In [ ]:
# Apply mapping to test set

rec_counts = {}
reports_with_recs = 0
reports_total = len(X_test)

for idx in range(reports_total):
    preds = {col: Y_pred[col][idx] for col in ["human_factors", "contributing_factors"]}
    probs = {col: Y_prob[col][idx] for col in ["human_factors", "contributing_factors"]}

    result = get_recommendations(preds, probs, confidence_threshold=0.5)

    if result["recommendations"]:
        reports_with_recs += 1

    for rec in result["recommendations"]:
        rec_counts[rec] = rec_counts.get(rec, 0) + 1

coverage = reports_with_recs / reports_total * 100

print(f"  Test reports with ≥1 recommendation : {reports_with_recs:,} / {reports_total:,} ({coverage:.1f}%)")
print(f"  Unique recommendations triggered    : {len(rec_counts)}")
print(f"\n  ── Top 15 most frequent recommendations ──")

sorted_recs = sorted(rec_counts.items(), key=lambda x: x[1], reverse=True)
for rec, count in sorted_recs[:15]:
    print(f"  {count:>5,}  ({count/reports_total*100:5.1f}%)  {rec}")

### 6.4 Pipeline Demonstration
- Below; run 5 sample narratives through the full pipeline — from raw text to predicted factors and preventive recommendations — to validate end-to-end behavior on realistic scenarios.

In [ ]:
# End-to-end demonstration on 5 sample narratives

import re

def demo_predict(narrative, idx, far_part="other", confidence_threshold=0.5):
    """Run full pipeline on a single narrative and print results."""

    # -------------------
    # Clean text
    # -------------------
    text = narrative.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # -------------------
    # Encode narrative
    # -------------------
    emb = encoder.encode([text])

    # -------------------
    # Encode FAR Part (safe handling)
    # -------------------
    if far_part not in ohe.categories_[0]:
        far_part = "other"

    struct_input = pd.DataFrame([{
        "aircraft1_far_part": far_part
    }])

    struct_encoded = ohe.transform(struct_input)

    # -------------------
    # Combine features
    # -------------------
    X = np.concatenate([emb, struct_encoded], axis=1)

    # -------------------
    # Predict
    # -------------------
    all_preds, all_probs = {}, {}

    for col in ["human_factors", "contributing_factors", "event_anomaly", "event_detector"]:
        label_names = mlb[col].classes_

        preds = np.zeros(len(label_names), dtype=int)
        probs = np.zeros(len(label_names))

        for i, label in enumerate(label_names):
            probs[i] = models[col][label].predict_proba(X)[:, 1][0]
            preds[i] = 1 if probs[i] >= confidence_threshold else 0

        all_preds[col] = preds
        all_probs[col] = probs

    # -------------------
    # Recommendations
    # -------------------
    recs = get_recommendations(all_preds, all_probs, confidence_threshold)

    # -------------------
    # Display
    # -------------------
    print(f"\n{'='*70}")
    print(f"  EXAMPLE {idx}")
    print(f"{'='*70}")
    print(f"  Narrative : {narrative[:150]}...")
    print(f"  Context   : FAR={far_part}")

    for col_label, col_key in [
        ("Human Factors", "human_factors"),
        ("Contributing Factors", "contributing_factors"),
        ("Event Anomaly", "event_anomaly"),
        ("Event Detector", "event_detector")
    ]:

        detected = [
            mlb[col_key].classes_[i]
            for i in range(len(mlb[col_key].classes_))
            if all_preds[col_key][i] == 1
        ]

        scores = {
            mlb[col_key].classes_[i]: round(float(all_probs[col_key][i]), 3)
            for i in range(len(mlb[col_key].classes_))
            if all_preds[col_key][i] == 1
        }

        if detected:
            print(f"\n  {col_label}:")
            for label, score in scores.items():
                print(f"    • {label:<45s} ({score:.3f})")
        else:
            print(f"\n  {col_label}: none above threshold")

    # -------------------
    # Recommendations output
    # -------------------
    if recs["recommendations"]:
        print(f"\n  Preventive Recommendations:")
        for i, rec in enumerate(recs["recommendations"], 1):
            print(f"    {i}. {rec}")
    else:
        print(f"\n  Preventive Recommendations: none generated")

    print()

In [ ]:
# 5 test scenarios

examples = [
    {
        "narrative": "During approach to runway 27L, the captain became distracted by a radio call from ATC and failed to monitor airspeed. The first officer noticed the deviation and called for a go-around. Crew coordination was lacking during the critical phase.",
        "far_part": "commercial airline"
    },
    {
        "narrative": "The maintenance technician was working a double shift and forgot to reinstall the oil cap after servicing the engine. The error was discovered during the next day's preflight inspection. Fatigue and time pressure contributed to the oversight.",
        "far_part": "commercial airline"
    },
    {
        "narrative": "ATC cleared two aircraft for simultaneous approaches to parallel runways. A miscommunication between the approach controller and the tower controller resulted in both aircraft being on the same frequency with conflicting instructions. The conflict was detected by TCAS.",
        "far_part": "commercial airline"
    },
    {
        "narrative": "While taxiing to the gate after landing, the first officer confused the taxiway signage and turned onto an active runway. The tower controller immediately issued a stop instruction. The captain had been heads-down reviewing the arrival paperwork.",
        "far_part": "commercial airline"
    },
    {
        "narrative": "A private pilot flying VFR encountered unexpected IMC conditions during a cross-country flight. The pilot had not received a formal weather briefing and attempted to descend below the cloud layer, resulting in a near terrain encounter. The pilot had limited instrument training.",
        "far_part": "general aviation"
    },
]

for i, ex in enumerate(examples, 1):
    demo_predict(
        narrative=ex["narrative"],
        idx=i,
        far_part=ex["far_part"]
    )

# ✔ Chapter Conclusion:
The preventive mapping layer successfully connects model predictions to actionable safety recommendations. Coverage reached 99.8% on the test set — nearly all reports receive at least one recommendation — confirming the mapping is practically useful. The recommendation frequency distribution reveals which interventions are most commonly triggered, providing operational insight into the dominant risk patterns in the dataset. This layer is designed for decision support — it highlights relevant safety actions, but does not replace expert judgment or causal investigation.

---
# **7.	Deployment**

This chapter packages the trained pipeline into a Gradio application deployed on Hugging Face Spaces. The app accepts a free-text incident narrative and optional structured fields, then returns predicted human factors, contributing factors, event anomaly, event detector, and actionable preventive recommendations.

---
## 7.1 Pipeline Export

\- Below; save all trained models, encoders, and binarizers required to reproduce the full inference pipeline.

In [ ]:
# Export pipeline artifacts

import pickle
import os

export_dir = "export"
os.makedirs(export_dir, exist_ok=True)

with open(f"{export_dir}/xgb_models.pkl", "wb") as f:
    pickle.dump(dict(models), f)

with open(f"{export_dir}/ohe_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)

with open(f"{export_dir}/mlb_binarizers.pkl", "wb") as f:
    pickle.dump(dict(mlb), f)

with open(f"{export_dir}/preventive_mappings.pkl", "wb") as f:
    pickle.dump({
        "human_factors": human_factors_mapping,
        "contributing_factors": contributing_factors_mapping,
    }, f)

with open(f"{export_dir}/struct_cols.pkl", "wb") as f:
    pickle.dump(struct_cols, f)

for fname in sorted(os.listdir(export_dir)):
    fsize = os.path.getsize(f"{export_dir}/{fname}") / 1024
    print(f"  {fname:<30s}  {fsize:>8.1f} KB")

In [ ]:
import shutil
shutil.make_archive("export", "zip", "export")
print("export.zip created")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp export.zip /content/drive/MyDrive/

- All pipeline components exported. These files are the only dependencies needed to run inference — no retraining or access to the original dataset is required.
- Note: xgb_models.pkl is ~52MB — within Hugging Face Spaces' free tier limits.

---
## 7.2 Core Artifacts

\- The trained classifiers and supporting objects are serialized for deployment.

- Core artifacts:
```
xgb_models.pkl          – 69 trained XGBoost binary classifiers (keyed by target + label)
ohe_encoder.pkl          – fitted OneHotEncoder for structured predictors
mlb_binarizers.pkl       – fitted MultiLabelBinarizers for all four targets
preventive_mappings.pkl  – rule-based human factors → recommendation lookup
struct_cols.pkl          – structured column names and order
```
- XGBoost models are exported using `pickle`. The MiniLM sentence transformer (`all-MiniLM-L6-v2`) is loaded at runtime directly from Hugging Face Hub — no local copy is needed.

---
## 7.3 Requirements File

- A `requirements.txt` file defines the runtime dependencies required by the Hugging Face Space:

```
gradio>=4.0
xgboost>=2.0
sentence-transformers>=2.2
scikit-learn>=1.3
numpy>=1.24
pandas>=2.0
```
- Hugging Face automatically installs these packages when the Space is initialized.


---
## 7.4 Repository Structure

- The final deployment layout for Hugging Face Spaces:

```
.
├── app.py
├── inference.py
├── export/
│   ├── xgb_models.pkl
│   ├── ohe_encoder.pkl
│   ├── mlb_binarizers.pkl
│   ├── preventive_mappings.pkl
│   └── struct_cols.pkl
├── requirements.txt
└── README.md
```
- Files exclusive to deployment:
  - inference.py contains the preprocessing, encoding, prediction, and recommendation logic,
  - app.py provides the Gradio interface that exposes the incident analysis system.
- The export/ directory contains all artifacts required for inference. (exported by this notebook)

---
## 7.5 Chapter Conclusion

This chapter demonstrated how the trained multi-label classification pipeline is transformed into a deployable application using Hugging Face Spaces. By exporting trained classifiers, persisting encoders and mapping objects, and defining a minimal runtime environment, the system transitions from a research notebook to a reproducible inference service. The full deployment implementation, including app.py and inference.py, is available in the Hugging Face Space repository.

At inference time, the app receives a raw narrative and optional FAR Part. The text is cleaned and encoded by MiniLM into a 384-dim vector, concatenated with the 4-dim OHE structured vector, passed to all 69 XGBoost classifiers, and predictions above 0.5 confidence are mapped to recommendations via the preventive mapping lookup.

**Note on training data scope:**
The deployed models were trained on the 70% training split and validated
on the 15% validation split. Retraining on the full dataset before
deployment was deliberately avoided — doing so would invalidate the
evaluation metrics reported in Chapter 5, as the deployed model would
have been trained on data that was previously held out. For a
proof-of-concept system at this scale (~37,600 rows), the marginal
performance gain from full-data retraining is negligible. A production
deployment would address this through a versioned retraining pipeline
with a dedicated held-out benchmark set maintained separately from the
training pool.

---
# **8.	Final Conclusions**

This project built an end-to-end pipeline for detecting human factors in aviation incident reports and mapping them to preventive recommendations. Starting from ~111K raw ASRS (Aviation Safety Reporting System) reports, the dataset was filtered to ~40K human-factor incidents and cleaned into a modeling-ready structure of 37,626 rows across 69 multi-label targets in four output dimensions.


Architecture & Features
A MiniLM (all-MiniLM-L6-v2) + XGBoost (Extreme Gradient Boosting) Binary Relevance architecture was trained with early stopping and SPW (scale_pos_weight) class-imbalance handling. Feature importance analysis confirmed that narrative embeddings drive 97.8% of predictive signal. The structured predictor aircraft1_far_part contributes 2.2% — small in absolute terms, but meaningful: general aviation (Part 91) and commercial airline (Part 121) operations have distinct human factor risk profiles, and this signal provides the model with operational context that the narrative alone may not always make explicit.


Evaluation
Evaluation on the held-out test set showed that event_anomaly and event_detector approached the 0.60 Macro F1 viability threshold (both at 0.58), while human_factors (0.36) and contributing_factors (0.37) fell short — consistent with the subjective, overlapping nature of human factor labeling in free-text reports. A rule-based preventive mapping layer achieved 99.8% recommendation coverage on the test set.


Deployment
The full system is deployed as a Gradio application on Hugging Face Spaces for use by aviation safety analysts. At inference time, a raw narrative and optional FAR Part are processed through the full pipeline — cleaning, MiniLM encoding, OHE structured encoding, 69-classifier prediction, and recommendation mapping — in a single request.


Limitations
The system is designed for decision support, not causal inference. Performance on human_factors and contributing_factors reflects the inherent ambiguity of the labels, not a modeling failure. Low-frequency labels are excluded from the mapping layer due to insufficient training support. The deployed models were trained on 70% of the data — full-dataset retraining would require a separate held-out benchmark to maintain evaluation integrity.